# 4. Mushroom foraging

The [mushroom dataset](https://www.kaggle.com/datasets/dhinaharp/mushroom-dataset) contains data about approximately 60000 mushrooms, and your task is to classify them as either edible or poisonous. You can read about the features [here](https://www.kaggle.com/datasets/uciml/mushroom-classification) and import the data using:

It's up to you how you approach this data, but at a minimum, your analysis should include:

* Informed **data preparation**.
* 2 different classification models, one of which must be **logistic regression**.
* A discussion of which **performance metric** is most relevant for the evaluation of your models.
* 2 different **validation methodologies** used to tune hyperparameters.
* **Confusion matrices** for your models, and associated comments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             recall_score, f1_score, precision_score)

SEED = 42

pd.set_option('display.max_columns', 1000)
df = pd.read_csv('secondary_data.csv', delimiter=';')
print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Overblik over datasættet
print('Klassefordeling (p = giftig, e = spiselig):')
print(df['class'].value_counts())

print('\nDatatyper:')
print(df.dtypes)

print('\nManglende værdier (kolonner med missing):')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nAntal unikke værdier pr. kolonne:')
print(df.nunique())

## Data Preparation

Datasættet har 61.069 svampe og 20 features. Der er ret mange manglende værdier i flere af kolonnerne, så jeg har valgt at håndtere det på følgende måde:

Kolonner som veil-type, veil-color, spore-print-color, stem-root, stem-surface og gill-spacing mangler alle over 40% af deres data. veil-type har desuden kun én unik værdi, så den giver ingen information overhovedet. De seks kolonner fjernes.

De resterende kolonner med manglende data (cap-surface, gill-attachment og ring-type) mangler under 25%, så dem fylder jeg ud med modus, altså den mest hyppige værdi i kolonnen.

Derefter one-hot encodes de kategoriske variable med `pd.get_dummies()`, og de tre numeriske features (cap-diameter, stem-height, stem-width) standardiseres med StandardScaler.

In [ ]:
# ── Trin 1: Fjern kolonner med for mange manglende værdier ─────────────────

drop_cols = ['veil-type', 'veil-color', 'spore-print-color',
             'stem-root', 'stem-surface', 'gill-spacing']
df_clean = df.drop(columns=drop_cols)

print(f'Fjernede {len(drop_cols)} kolonner: {drop_cols}')
print(f'Resterende shape: {df_clean.shape}')

In [ ]:
# ── Trin 2: Imputér resterende manglende værdier med modus ─────────────────

impute_cols = ['cap-surface', 'gill-attachment', 'ring-type']
for col in impute_cols:
    mode_val = df_clean[col].mode()[0]
    n_missing = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(mode_val)
    print(f'{col}: imputerede {n_missing} manglende med modus = "{mode_val}"')

print(f'\nResterende manglende værdier: {df_clean.isnull().sum().sum()}')

In [ ]:
# ── Trin 3: Encode target og features ──────────────────────────────────────

# Target: p (poisonous) = 1, e (edible) = 0
y = (df_clean['class'] == 'p').astype(int).values
print(f'Target: poisonous (1): {y.sum()}, edible (0): {(y == 0).sum()}')

# One-hot encode kategoriske features
X = df_clean.drop(columns=['class'])
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

print(f'\nNumeriske features ({len(num_cols)}): {num_cols}')
print(f'Kategoriske features ({len(cat_cols)}): {cat_cols}')

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f'\nFeature matrix shape efter one-hot encoding: {X.shape}')

In [ ]:
# ── Trin 4: Train/test split og skalering ──────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Standardisér numeriske features (vigtigt for logistisk regression)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test[num_cols])

print(f'X_train: {X_train_scaled.shape}')
print(f'X_test:  {X_test_scaled.shape}')
print(f'\ny_train fordeling: poisonous={y_train.sum()}, edible={(y_train==0).sum()}')
print(f'y_test fordeling:  poisonous={y_test.sum()}, edible={(y_test==0).sum()}')

## Valg af performance metric

Når man klassificerer svampe som spiselige eller giftige, er det vigtigt at tænke over hvad de forskellige fejltyper betyder i praksis.

En false negative betyder at en giftig svamp bliver klassificeret som spiselig, hvilket potentielt er livstruende. En false positive betyder bare at man kasserer en spiselig svamp, hvilket er harmløst.

Derfor er recall for den giftige klasse den vigtigste metrik her. Recall måler hvor stor en andel af de faktisk giftige svampe modellen fanger:

$$\text{Recall} = \frac{TP}{TP + FN}$$

En recall på 1.0 ville betyde at alle giftige svampe fanges. Vi vil hellere kassere nogle spiselige svampe end at overse en giftig.

Jeg bruger også accuracy og F1-score til at give et samlet billede, men recall for poisonous er den metrik jeg primært evaluerer modellerne på.

## Model 1: Logistic Regression

Logistisk regression estimerer sandsynligheden for at en observation tilhører en given klasse. Jeg tuner hyperparameteren C (inverse regulariseringsstyrke) med GridSearchCV.

Valideringsmetodologi 1: GridSearchCV med 5-fold stratified cross-validation. Træningsdata opdeles i 5 folds, og modellen trænes på 4 og valideres på den sidste, gentaget for alle folds og alle C-værdier. Den bedste C vælges ud fra gennemsnitlig recall.

In [ ]:
# ── Logistic Regression – GridSearchCV ─────────────────────────────────────

param_grid_lr = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=5000, random_state=SEED, solver='lbfgs'),
    param_grid=param_grid_lr,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='recall',       # optimerer for recall (giftige svampe)
    return_train_score=True,
    n_jobs=-1
)

lr_grid.fit(X_train_scaled, y_train)

print('GridSearchCV resultater (Logistic Regression):')
results_lr = pd.DataFrame(lr_grid.cv_results_)[
    ['param_C', 'mean_train_score', 'mean_test_score', 'std_test_score']
]
results_lr.columns = ['C', 'Train Recall (mean)', 'Val Recall (mean)', 'Val Recall (std)']
print(results_lr.to_string(index=False))

print(f'\nBedste C: {lr_grid.best_params_["C"]}')
print(f'Bedste CV recall: {lr_grid.best_score_:.4f}')

In [ ]:
# ── Evaluér bedste Logistic Regression på test-data ────────────────────────

best_lr = lr_grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_scaled)

print(f'─── Logistic Regression (C={lr_grid.best_params_["C"]}) – Test Set ───')
print(f'Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_lr):.4f}')
print(f'F1-score : {f1_score(y_test, y_pred_lr):.4f}')
print('\nKlassifikationsrapport:')
print(classification_report(y_test, y_pred_lr,
                            target_names=['Edible (0)', 'Poisonous (1)']))

## Model 2: Random Forest

Random Forest er en ensemble-metode der bygger mange beslutningstræer og kombinerer deres predictions via majority voting. Den kan fange ikke-lineære sammenhænge og interaktioner mellem features bedre end logistisk regression.

Valideringsmetodologi 2: Manuel 10-fold stratified cross-validation med `cross_val_score`. Jeg tester forskellige kombinationer af n_estimators (antal træer) og max_depth (dybde af træerne) og vælger den konfiguration der giver højest recall.

In [ ]:
# ── Random Forest – Manuel 10-fold CV ──────────────────────────────────────

cv_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

rf_configs = [
    {'n_estimators': 50,  'max_depth': 10},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 100, 'max_depth': 20},
    {'n_estimators': 200, 'max_depth': 20},
    {'n_estimators': 200, 'max_depth': None},
]

rf_results = []
for cfg in rf_configs:
    rf = RandomForestClassifier(**cfg, random_state=SEED, n_jobs=-1)
    scores = cross_val_score(rf, X_train_scaled, y_train,
                             cv=cv_10, scoring='recall')
    rf_results.append({
        'n_estimators': cfg['n_estimators'],
        'max_depth': cfg['max_depth'],
        'Recall (mean)': round(scores.mean(), 4),
        'Recall (std)': round(scores.std(), 4)
    })

rf_cv_df = pd.DataFrame(rf_results)
print('10-fold CV resultater (Random Forest):')
print(rf_cv_df.to_string(index=False))

best_rf_cfg = max(rf_results, key=lambda x: x['Recall (mean)'])
print(f'\nBedste konfiguration: n_estimators={best_rf_cfg["n_estimators"]}, '
      f'max_depth={best_rf_cfg["max_depth"]}')

In [ ]:
# ── Træn bedste Random Forest og evaluér på test-data ──────────────────────

best_rf = RandomForestClassifier(
    n_estimators=best_rf_cfg['n_estimators'],
    max_depth=best_rf_cfg['max_depth'],
    random_state=SEED, n_jobs=-1
)
best_rf.fit(X_train_scaled, y_train)
y_pred_rf = best_rf.predict(X_test_scaled)

print(f'─── Random Forest (n_estimators={best_rf_cfg["n_estimators"]}, '
      f'max_depth={best_rf_cfg["max_depth"]}) – Test Set ───')
print(f'Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_rf):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_rf):.4f}')
print(f'F1-score : {f1_score(y_test, y_pred_rf):.4f}')
print('\nKlassifikationsrapport:')
print(classification_report(y_test, y_pred_rf,
                            target_names=['Edible (0)', 'Poisonous (1)']))

## Confusion Matrices

In [ ]:
# Confusion Matrices

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = ['Edible (0)', 'Poisonous (1)']

cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=labels).plot(
    ax=axes[0], cmap='Blues', values_format='d'
)
axes[0].set_title(f'Logistic Regression (C={lr_grid.best_params_["C"]})')

cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=labels).plot(
    ax=axes[1], cmap='Greens', values_format='d'
)
axes[1].set_title(f'Random Forest (n={best_rf_cfg["n_estimators"]}, '
                  f'depth={best_rf_cfg["max_depth"]})')

plt.tight_layout()
plt.show()

for name, cm in [('Logistic Regression', cm_lr), ('Random Forest', cm_rf)]:
    tn, fp, fn, tp = cm.ravel()
    print(f'\n{name}:')
    print(f'  True Negatives  (edible korrekt):    {tn}')
    print(f'  False Positives (edible som giftig):  {fp}')
    print(f'  False Negatives (giftig som edible):  {fn}')
    print(f'  True Positives  (giftig korrekt):     {tp}')

### Kommentarer til confusion matrices

Logistic Regression har 1.278 false negatives, altså giftige svampe der bliver klassificeret som spiselige. Det er ret mange. Derudover er der 2.201 false positives. Recall for poisonous lander på 81,1%, hvilket vil sige at ca. 19% af de giftige svampe slipper igennem. Det er ikke godt nok i en situation hvor det handler om liv og død. Modellen er begrænset af sin lineære beslutningsgrænse og kan ikke rigtig fange de komplekse mønstre i dataene.

Random Forest har derimod kun 2 false negatives ud af 6.778 giftige svampe, og 7 false positives. Recall for poisonous er 99,97%, så den fanger næsten alle giftige svampe. Det skyldes at modellen kan modellere ikke-lineære sammenhænge via sine mange beslutningstræer.

Samlet set er Random Forest klart den bedste model her. Med kun 2 false negatives mod 1.278 for logistisk regression er den markant sikrere. Hvis man faktisk skulle bruge en model til at vurdere svampe, ville Random Forest være det oplagte valg.

## Sammenligning af valideringsmetodologier

Til logistisk regression brugte jeg GridSearchCV med 5-fold stratified CV. Det er en automatisk tilgang der systematisk tester alle C-værdier og vælger den bedste. Fordelen er at det er kompakt og nemt at bruge, men det kan blive langsomt hvis man har mange hyperparametre.

Til Random Forest brugte jeg i stedet `cross_val_score` med 10-fold stratified CV, hvor jeg manuelt definerede de konfigurationer jeg ville teste. Det giver lidt mere kontrol, og med 10 folds i stedet for 5 får man et mere stabilt estimat med lavere varians, men det tager også længere tid.

Begge metoder bruger stratified folds, som sikrer at klassefordelingen (edible/poisonous) bevares i hvert fold. Det er vigtigt her fordi klasserne ikke er helt lige fordelt.